In [6]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '../../src/serving')

import time
import torch
import joblib
import numpy as np
import json
from pathlib import Path

# Import service directly
from bentoml_service import (
    model, user_seqs,
    token2movie, title_map,
    PAD_TOKEN, vocab,
    PROC
)

print("✅ BentoML service module imported")
print(f"   Vocab size : {vocab['vocab_size']}")
print(f"   Users      : {len(user_seqs):,}")
print(f"   Titles     : {len(title_map):,}")

✅ BentoML service module imported
   Vocab size : 9069
   Users      : 670
   Titles     : 45,454


In [8]:
def test_recommend(user_id: int,
                   top_k: int = 10):
    """Test recommendation logic directly"""
    import pandas as pd

    start = time.time()
    seq   = user_seqs.get(user_id, [])

    if not seq:
        print(f"⚠️  User {user_id} "
              f"not in sequences")
        return [], 0

    pad_l = 50 - len(seq)
    hist  = torch.LongTensor(
        [[PAD_TOKEN]*pad_l + seq[-50:]])

    with torch.no_grad():
        toks, scores = model.predict(
            hist,
            top_k=min(top_k * 10, 500))

    rated    = set(seq)
    seen_mid = set()
    recs     = []

    sc_arr   = scores[0].cpu().numpy()
    sc_min   = sc_arr.min()
    sc_max   = sc_arr.max()
    sc_range = max(sc_max - sc_min, 1e-6)

    for tok, sc in zip(
            toks[0].cpu().numpy(), sc_arr):
        mid = token2movie.get(int(tok))
        if not mid: continue
        if int(tok) in rated: continue
        if mid in seen_mid: continue

        seen_mid.add(mid)
        title = title_map.get(
            mid, f"Movie {mid}")
        norm  = float(
            (sc - sc_min) / sc_range)

        recs.append({
            "rank":       len(recs)+1,
            "movie_id":   int(mid),
            "title":      str(title)[:40],
            "score":      round(norm, 4),
            "cold_start": False,
            "fallback":   False,
        })

        if len(recs) >= top_k:
            break

    # Popularity padding
    if len(recs) < top_k:
        try:
            ratings_df = pd.read_csv(
                PROC / 'ratings_cleaned.csv')
            pop = ratings_df.groupby(
                'movieId')['rating']\
                .count()\
                .sort_values(ascending=False)

            rated_mids = set(
                token2movie.get(int(t), -1)
                for t in seq)

            for mid in pop.index:
                if len(recs) >= top_k: break
                mid = int(mid)
                if mid not in rated_mids \
                   and mid not in seen_mid:
                    t = title_map.get(
                        mid, f"Movie {mid}")
                    seen_mid.add(mid)
                    recs.append({
                        "rank":       len(recs)+1,
                        "movie_id":   mid,
                        "title":      str(t)[:40],
                        "score":      0.001,
                        "cold_start": False,
                        "fallback":   True,
                    })
        except Exception as e:
            print(f"Padding error: {e}")

    # Re-number ranks
    for i, r in enumerate(recs):
        r['rank'] = i + 1

    latency = (time.time()-start)*1000
    return recs, latency


# Test on 3 users
test_users = [1, 481, 196]

for uid in test_users:
    recs, latency = test_recommend(
        uid, top_k=10)
    print(f"\nUser {uid} "
          f"({latency:.1f}ms) "
          f"— {len(recs)} recs:")
    for r in recs[:5]:
        fb = "📌" if r['fallback'] else "🎯"
        print(f"  {fb} {r['rank']}. "
              f"{r['title']:<40} "
              f"{r['score']:.4f}")


User 1 (233.1ms) — 10 recs:
  🎯 1. Heat                                     0.0521
  🎯 2. Father of the Bride Part II              0.0373
  🎯 3. Waiting to Exhale                        0.1106
  🎯 4. Grumpier Old Men                         0.8457
  📌 5. Forrest Gump                             0.0010

User 481 (44.4ms) — 10 recs:
  🎯 1. Heat                                     0.0373
  🎯 2. Father of the Bride Part II              0.0521
  🎯 3. Waiting to Exhale                        0.8218
  🎯 4. Grumpier Old Men                         0.0452
  📌 5. Forrest Gump                             0.0010

User 196 (35.8ms) — 10 recs:
  🎯 1. Heat                                     0.0621
  🎯 2. Waiting to Exhale                        0.3794
  🎯 3. Grumpier Old Men                         0.0807
  📌 4. Forrest Gump                             0.0010
  📌 5. Pulp Fiction                             0.0010


In [9]:
print("LATENCY BENCHMARK")
print("=" * 45)

N   = 30
uid = 481
lats_direct = []

for _ in range(N):
    _, lat = test_recommend(uid, 10)
    lats_direct.append(lat)

print(f"Direct PyTorch inference (n={N})")
print(f"  p50 : "
      f"{np.percentile(lats_direct,50):.1f}ms")
print(f"  p95 : "
      f"{np.percentile(lats_direct,95):.1f}ms")
print(f"  p99 : "
      f"{np.percentile(lats_direct,99):.1f}ms")
print(f"  mean: "
      f"{np.mean(lats_direct):.1f}ms")

sla_ok = np.percentile(lats_direct,99) < 100
print(f"\nSLA p99 < 100ms: "
      f"{'✅' if sla_ok else '⚠️'}")

LATENCY BENCHMARK
Direct PyTorch inference (n=30)
  p50 : 38.7ms
  p95 : 107.9ms
  p99 : 142.3ms
  mean: 51.0ms

SLA p99 < 100ms: ⚠️


In [10]:
print("ONNX EXPORT")
print("=" * 45)

import os
Path('../../models/onnx').mkdir(
    parents=True, exist_ok=True)

onnx_path = '../../models/onnx/'\
            'hstu_serving.onnx'

dummy_hist = torch.LongTensor(
    [[1,2,3,4,5] + [PAD_TOKEN]*45])
dummy_pos  = torch.LongTensor([6])
dummy_neg  = torch.LongTensor([7])

try:
    torch.onnx.export(
        model,
        (dummy_hist, dummy_pos, dummy_neg),
        onnx_path,
        export_params = True,
        opset_version = 17,
        input_names   = ['history',
                         'pos_items',
                         'neg_items'],
        output_names  = ['pos_scores',
                         'neg_scores',
                         'rating_pred',
                         'completion_pred'],
        dynamic_axes  = {
            'history': {0: 'batch_size'}},
        verbose=False)

    size = os.path.getsize(onnx_path)/1e6
    print(f"✅ ONNX exported: {size:.1f}MB")

    # ONNX vs PyTorch latency
    import onnxruntime as ort

    sess = ort.InferenceSession(
        onnx_path,
        providers=[
            'CPUExecutionProvider'])

    N       = 20
    t_pt    = []
    t_onnx  = []

    for _ in range(N):
        s = time.time()
        with torch.no_grad():
            model.encode_user(dummy_hist)
        t_pt.append(
            (time.time()-s)*1000)

        s = time.time()
        sess.run(None, {
            'history':   dummy_hist.numpy(),
            'pos_items': dummy_pos.numpy(),
            'neg_items': dummy_neg.numpy(),
        })
        t_onnx.append(
            (time.time()-s)*1000)

    speedup = np.median(t_pt) / \
              max(np.median(t_onnx), 0.01)

    print(f"\nPyTorch p50 : "
          f"{np.percentile(t_pt,50):.1f}ms")
    print(f"ONNX p50    : "
          f"{np.percentile(t_onnx,50):.1f}ms")
    print(f"Speedup     : {speedup:.1f}x")

except Exception as e:
    print(f"⚠️  ONNX: {e}")
    print("   Continuing without ONNX")
    speedup = 1.0

ONNX EXPORT
⚠️  ONNX: Module [HSTURanker] is missing the required "forward" function
   Continuing without ONNX


In [11]:
print("BENTOML SERVICE RESULTS")
print("=" * 45)
print("Verified via curl in terminal:\n")

bentoml_results = {
    "health": {
        "status":     "healthy",
        "vocab_size": 9069,
        "n_users":    670,
        "n_titles":   45454,
    },
    "recommend": {
        "user_id":    481,
        "n_recs":     10,
        "latency_ms": 45.01,
        "sample": [
            "Heat",
            "Father of the Bride Part II",
            "Waiting to Exhale",
            "Grumpier Old Men",
            "Forrest Gump (fallback)",
        ],
    },
    "feedback": {
        "status":   "logged",
        "user_id":  1,
        "movie_id": 356,
        "rating":   4.5,
    },
}

for endpoint, data in \
        bentoml_results.items():
    print(f"  /{endpoint} ✅")
    for k, v in data.items():
        print(f"    {k}: {v}")
    print()

BENTOML SERVICE RESULTS
Verified via curl in terminal:

  /health ✅
    status: healthy
    vocab_size: 9069
    n_users: 670
    n_titles: 45454

  /recommend ✅
    user_id: 481
    n_recs: 10
    latency_ms: 45.01
    sample: ['Heat', 'Father of the Bride Part II', 'Waiting to Exhale', 'Grumpier Old Men', 'Forrest Gump (fallback)']

  /feedback ✅
    status: logged
    user_id: 1
    movie_id: 356
    rating: 4.5



In [12]:
day29_results = {
    "service":   "BentoML 1.x",
    "model":     "HSTU",
    "port":      3001,
    "endpoints": {
        "/health":    "✅ working",
        "/recommend": "✅ working",
        "/feedback":  "✅ working",
    },
    "curl_format": {
        "note":    "BentoML 1.x wraps "
                   "body in request key",
        "example": '{"request": '
                   '{"user_id": 481}}',
    },
    "direct_inference": {
        "p50_ms": round(float(
            np.percentile(
                lats_direct, 50)), 1),
        "p95_ms": round(float(
            np.percentile(
                lats_direct, 95)), 1),
        "p99_ms": round(float(
            np.percentile(
                lats_direct, 99)), 1),
        "sla_met": bool(sla_ok),
    },
    "bentoml_latency_ms": 45.01,
    "features": [
        "pydantic_input_validation",
        "deduplication",
        "normalized_scores",
        "popularity_padding",
        "cold_start_fallback",
        "feedback_endpoint",
    ],
}

PROC_PATH = '../../data/processed/'
with open(
        PROC_PATH + 'day29_results.json',
        'w') as f:
    json.dump(day29_results, f, indent=2)

print("✅ Day 29 results saved")
print(json.dumps(day29_results, indent=2))

✅ Day 29 results saved
{
  "service": "BentoML 1.x",
  "model": "HSTU",
  "port": 3001,
  "endpoints": {
    "/health": "\u2705 working",
    "/recommend": "\u2705 working",
    "/feedback": "\u2705 working"
  },
  "curl_format": {
    "note": "BentoML 1.x wraps body in request key",
    "example": "{\"request\": {\"user_id\": 481}}"
  },
  "direct_inference": {
    "p50_ms": 38.7,
    "p95_ms": 107.9,
    "p99_ms": 142.3,
    "sla_met": false
  },
  "bentoml_latency_ms": 45.01,
  "features": [
    "pydantic_input_validation",
    "deduplication",
    "normalized_scores",
    "popularity_padding",
    "cold_start_fallback",
    "feedback_endpoint"
  ]
}
